In [4]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window


StatementMeta(, 219a55af-906e-4fc8-ac0d-a6c272e67222, 16, Finished, Available, Finished)

In [6]:
patients_path = "Files/patients.csv"
staff_path = "Files/staff.csv"
staff_schedule_path = "Files/staff_schedule.csv"
services_weekly_path = "Files/services_weekly.csv"

StatementMeta(, 219a55af-906e-4fc8-ac0d-a6c272e67222, 18, Finished, Available, Finished)

In [9]:
patients_df_raw = (
    spark.read
         .option("header", "true")
         .option("inferSchema", "true")
         .csv(patients_path)
)

patients_df = (
    patients_df_raw
    .withColumn("age", F.col("age").cast("int"))
    .withColumn("arrival_date", F.col("arrival_date").cast("date"))
    .withColumn("departure_date", F.col("departure_date").cast("date"))
    .withColumn("satisfaction", F.col("satisfaction").cast("double"))
)

patients_df.printSchema()

(
    patients_df
        .write
        .mode("overwrite")
        .format("delta")
        .saveAsTable("patients")
)

print("Table 'patients' created.")


StatementMeta(, 219a55af-906e-4fc8-ac0d-a6c272e67222, 21, Finished, Available, Finished)

root
 |-- patient_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- arrival_date: date (nullable = true)
 |-- departure_date: date (nullable = true)
 |-- service: string (nullable = true)
 |-- satisfaction: double (nullable = true)

Table 'patients' created.


In [ ]:
staff_df_raw = (
    spark.read
         .option("header", "true")
         .option("inferSchema", "true")
         .csv(staff_path)
)

staff_df = staff_df_raw

staff_df.printSchema()

(
    staff_df
        .write
        .mode("overwrite")
        .format("delta")
        .saveAsTable("staff")
)

print("Table 'staff' created.")


StatementMeta(, 219a55af-906e-4fc8-ac0d-a6c272e67222, 22, Finished, Available, Finished)

root
 |-- staff_id: string (nullable = true)
 |-- staff_name: string (nullable = true)
 |-- role: string (nullable = true)
 |-- service: string (nullable = true)

Table 'staff' created.


In [11]:
staff_schedule_df_raw = (
    spark.read
         .option("header", "true")
         .option("inferSchema", "true")
         .csv(staff_schedule_path)
)

staff_schedule_df = (
    staff_schedule_df_raw
    .withColumn("week", F.col("week").cast("int"))
    .withColumn("present", F.col("present").cast("int"))
)

staff_schedule_df.printSchema()

(
    staff_schedule_df
        .write
        .mode("overwrite")
        .format("delta")
        .saveAsTable("staff_schedule")
)

print("Table 'staff_schedule' created.")


StatementMeta(, 219a55af-906e-4fc8-ac0d-a6c272e67222, 23, Finished, Available, Finished)

root
 |-- week: integer (nullable = true)
 |-- staff_id: string (nullable = true)
 |-- staff_name: string (nullable = true)
 |-- role: string (nullable = true)
 |-- service: string (nullable = true)
 |-- present: integer (nullable = true)

Table 'staff_schedule' created.


In [13]:
services_weekly_df_raw = (
    spark.read
         .option("header", "true")
         .option("inferSchema", "true")
         .csv(services_weekly_path)
)

services_weekly_df = (
    services_weekly_df_raw
    .withColumn("week", F.col("week").cast("int"))
    .withColumn("month", F.col("month").cast("int"))
    .withColumn("available_beds", F.col("available_beds").cast("int"))
    .withColumn("patients_request", F.col("patients_request").cast("int"))
    .withColumn("patients_admitted", F.col("patients_admitted").cast("int"))
    .withColumn("patients_refused", F.col("patients_refused").cast("int"))
    .withColumn("patient_satisfaction", F.col("patient_satisfaction").cast("double"))
    .withColumn("staff_morale", F.col("staff_morale").cast("double"))
)

services_weekly_df.printSchema()
services_weekly_df.show(10)

(
    services_weekly_df
        .write
        .mode("overwrite")
        .format("delta")
        .saveAsTable("services_weekly")
)

print("Table 'services_weekly' created.")


StatementMeta(, 219a55af-906e-4fc8-ac0d-a6c272e67222, 25, Finished, Available, Finished)

root
 |-- week: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- service: string (nullable = true)
 |-- available_beds: integer (nullable = true)
 |-- patients_request: integer (nullable = true)
 |-- patients_admitted: integer (nullable = true)
 |-- patients_refused: integer (nullable = true)
 |-- patient_satisfaction: double (nullable = true)
 |-- staff_morale: double (nullable = true)
 |-- event: string (nullable = true)

+----+-----+----------------+--------------+----------------+-----------------+----------------+--------------------+------------+--------+
|week|month|         service|available_beds|patients_request|patients_admitted|patients_refused|patient_satisfaction|staff_morale|   event|
+----+-----+----------------+--------------+----------------+-----------------+----------------+--------------------+------------+--------+
|   1|    1|       emergency|            32|              76|               32|              44|                67.0|        70.0|  

In [14]:
busiest_services = spark.sql("""
SELECT
    service,
    COUNT(*) AS num_patients,
    ROUND(AVG(satisfaction), 2) AS avg_satisfaction
FROM patients
GROUP BY service
ORDER BY num_patients DESC
""")

busiest_services.show()


StatementMeta(, 219a55af-906e-4fc8-ac0d-a6c272e67222, 26, Finished, Available, Finished)

+----------------+------------+----------------+
|         service|num_patients|avg_satisfaction|
+----------------+------------+----------------+
|       emergency|         263|           79.55|
|         surgery|         254|           80.31|
|general_medicine|         242|           78.57|
|             ICU|         241|           79.92|
+----------------+------------+----------------+



In [15]:
staff_load_by_week = spark.sql("""
SELECT
    week,
    COUNT(DISTINCT staff_id) AS num_staff,
    SUM(present) AS total_presence
FROM staff_schedule
GROUP BY week
HAVING COUNT(DISTINCT staff_id) >= 5
ORDER BY total_presence DESC
""")

staff_load_by_week.show()


StatementMeta(, 219a55af-906e-4fc8-ac0d-a6c272e67222, 27, Finished, Available, Finished)

+----+---------+--------------+
|week|num_staff|total_presence|
+----+---------+--------------+
|  26|      126|           119|
|  19|      126|           118|
|  41|      126|           118|
|  49|      126|           118|
|  23|      126|           117|
|  28|      126|           116|
|   1|      126|           116|
|  38|      126|           116|
|  22|      126|           115|
|  20|      126|           115|
|   7|      126|           115|
|  29|      126|           115|
|   2|      126|           115|
|  16|      126|           113|
|  34|      126|           112|
|  44|      126|           112|
|  47|      126|           112|
|  43|      126|           112|
|  10|      126|           112|
|  50|      126|           112|
+----+---------+--------------+
only showing top 20 rows



In [16]:
critical_event_weeks = spark.sql("""
WITH weekly_services AS (
    SELECT
        week,
        SUM(patients_request) AS total_requests,
        SUM(patients_admitted) AS total_admitted,
        SUM(patients_refused) AS total_refused,
        ROUND(AVG(patient_satisfaction), 2) AS avg_service_satisfaction,
        MAX(event) AS event
    FROM services_weekly
    GROUP BY week
),

weekly_patients AS (
    SELECT
        WEEKOFYEAR(arrival_date) AS week,
        COUNT(*) AS num_patients,
        ROUND(AVG(satisfaction), 2) AS avg_patient_satisfaction
    FROM patients
    GROUP BY WEEKOFYEAR(arrival_date)
)

SELECT
    ws.week,
    ws.total_requests,
    ws.total_admitted,
    ws.total_refused,
    ws.avg_service_satisfaction,
    ws.event,
    wp.num_patients,
    wp.avg_patient_satisfaction,
    (ws.total_requests - ws.total_admitted) AS pressure_score
FROM weekly_services ws
LEFT JOIN weekly_patients wp
    ON ws.week = wp.week
WHERE ws.total_requests > 250
   OR ws.event = 'flu'
ORDER BY ws.week ASC
""")

critical_event_weeks.show()


StatementMeta(, 219a55af-906e-4fc8-ac0d-a6c272e67222, 28, Finished, Available, Finished)

+----+--------------+--------------+-------------+------------------------+------+------------+------------------------+--------------+
|week|total_requests|total_admitted|total_refused|avg_service_satisfaction| event|num_patients|avg_patient_satisfaction|pressure_score|
+----+--------------+--------------+-------------+------------------------+------+------------+------------------------+--------------+
|   1|           438|           136|          302|                   82.75|  none|          22|                   83.32|           302|
|   2|           385|           104|          281|                   80.75|  none|          18|                   75.94|           281|
|   3|           322|           116|          206|                   78.25|  none|          26|                    79.5|           206|
|   4|           387|           151|          236|                    72.0|  none|          24|                   80.75|           236|
|   5|           574|           104|          47

In [17]:
busiest_services.write.mode("overwrite").saveAsTable("busiest_services")
staff_load_by_week.write.mode("overwrite").saveAsTable("staff_load_by_week")
critical_event_weeks.write.mode("overwrite").saveAsTable("critical_event_weeks")


StatementMeta(, 219a55af-906e-4fc8-ac0d-a6c272e67222, 29, Finished, Available, Finished)

In [18]:
%%sql
SELECT * FROM busiest_services;

StatementMeta(, 219a55af-906e-4fc8-ac0d-a6c272e67222, 30, Finished, Available, Finished)

<Spark SQL result set with 4 rows and 3 fields>

In [19]:
%%sql
SELECT * FROM staff_load_by_week;

StatementMeta(, 219a55af-906e-4fc8-ac0d-a6c272e67222, 31, Finished, Available, Finished)

<Spark SQL result set with 52 rows and 3 fields>

In [20]:
%%sql
SELECT * FROM critical_event_weeks;


StatementMeta(, 219a55af-906e-4fc8-ac0d-a6c272e67222, 32, Finished, Available, Finished)

<Spark SQL result set with 22 rows and 9 fields>